In [1]:
from pathlib import Path
import pandas as pd

# Works when the notebook runs from the project root or notebooks folder.
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

df = pd.read_excel(root / "data" / "raw" / "Online Retail.xlsx")

print(df.shape)
df.head()

(541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [2]:
df.info()

print("\nMissing values:")
print(df.isna().sum())

print("\nExact duplicate rows:", df.duplicated().sum())

print("\nDate range:")
print(df["InvoiceDate"].min(), "to", df["InvoiceDate"].max())

print("\nCancelled invoice rows:")
print(
    df["InvoiceNo"].astype(str).str.upper().str.startswith("C").sum()
)

print("\nRows with non-positive quantities:")
print((df["Quantity"] <= 0).sum())

print("\nRows with non-positive prices:")
print((df["UnitPrice"] <= 0).sum())

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 33.1+ MB

Missing values:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Exact duplicate rows: 5268

Date range:
2010-12-01 08:26:00 to 2011-12-09 12:50:00

Cancelled invoice ro

In [3]:
clean = df.copy()
audit = []

def remove_rows(data, mask, reason):
    audit.append({
        "reason": reason,
        "rows_removed": int(mask.sum())
    })
    return data.loc[~mask].copy()

clean = remove_rows(
    clean,
    clean["InvoiceNo"].astype(str).str.upper().str.startswith("C"),
    "Cancelled invoices"
)

clean = remove_rows(
    clean,
    clean["Quantity"] <= 0,
    "Non-positive quantities"
)

clean = remove_rows(
    clean,
    clean["UnitPrice"] <= 0,
    "Non-positive prices"
)

clean = remove_rows(
    clean,
    clean[["StockCode", "InvoiceDate", "Quantity"]].isna().any(axis=1),
    "Missing essential fields"
)

clean["StockCode"] = clean["StockCode"].astype(str).str.strip()

cleaning_report = pd.DataFrame(audit)
display(cleaning_report)

print("Original rows:", len(df))
print("Remaining rows:", len(clean))

,reason,rows_removed
0,Cancelled invoices,9288
1,Non-positive quantities,1336
2,Non-positive prices,1181
3,Missing essential fields,0


Original rows: 541909
Remaining rows: 530104


In [4]:
display(
    clean.loc[
        ~clean["StockCode"].str.match(r"^\d"),
        ["StockCode", "Description"]
    ].drop_duplicates().sort_values("StockCode")
)

,StockCode,Description
15017,AMAZONFEE,AMAZON FEE
299982,B,Adjust bad debt
4406,BANK CHARGES,Bank Charges
1423,C2,CARRIAGE
24906,DCGS0003,BOXED GLASS ASHTRAY
170783,DCGS0004,HAYNES CAMPER SHOULDER BAG
75053,DCGS0069,OOH LA LA DOGS COLLAR
40052,DCGS0070,CAMOUFLAGE DOG COLLAR
21326,DCGS0076,SUNJAR LED NIGHT NIGHT LIGHT
84016,DCGSSBOY,BOYS PARTY BAG


In [5]:
exclude_codes = {
    "AMAZONFEE", "B", "BANK CHARGES",
    "C2", "DOT", "POST", "M", "S"
}

# Normalise only for comparison; preserve the original product codes.
codes = clean["StockCode"].str.upper()

exclude_mask = (
    codes.isin(exclude_codes)
    | codes.str.startswith("GIFT_")
)

# Keep excluded entries for inspection.
excluded_entries = clean.loc[exclude_mask].copy()

clean = remove_rows(
    clean,
    exclude_mask,
    "Fees, postage, vouchers, samples and manual entries"
)

cleaning_report = pd.DataFrame(audit)
display(cleaning_report)
print("Remaining rows:", len(clean))

,reason,rows_removed
0,Cancelled invoices,9288
1,Non-positive quantities,1336
2,Non-positive prices,1181
3,Missing essential fields,0
4,"Fees, postage, vouchers, samples and manual en...",2343


Remaining rows: 527761


In [6]:
output_dir = root / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

clean.to_csv(output_dir / "cleaned_transactions.csv", index=False)
cleaning_report.to_csv(output_dir / "cleaning_report.csv", index=False)
excluded_entries.to_csv(output_dir / "excluded_entries.csv", index=False)

In [7]:
# Identify the complete weeks covered by the original dataset.
first_date = df["InvoiceDate"].min().normalize()
last_date = df["InvoiceDate"].max().normalize()

first_monday = first_date - pd.Timedelta(days=first_date.weekday())
if first_date != first_monday:
    first_monday += pd.Timedelta(weeks=1)

last_monday = last_date - pd.Timedelta(days=last_date.weekday())
if last_date < last_monday + pd.Timedelta(days=6):
    last_monday -= pd.Timedelta(weeks=1)

weeks = pd.date_range(first_monday, last_monday, freq="W-MON")

# Label each transaction with the Monday starting its week.
sales = clean.copy()
sales["week_start"] = (
    sales["InvoiceDate"]
    .dt.to_period("W-SUN")
    .dt.start_time
)

sales = sales.loc[sales["week_start"].isin(weeks)]

weekly_sales = (
    sales.groupby(["StockCode", "week_start"])["Quantity"]
    .sum()
    .rename("units")
)

# Include weeks where a product has no recorded sales.
grid = pd.MultiIndex.from_product(
    [sorted(sales["StockCode"].unique()), weeks],
    names=["StockCode", "week_start"]
)

weekly = (
    weekly_sales.reindex(grid, fill_value=0)
    .reset_index()
    .sort_values(["StockCode", "week_start"])
)

display(weekly.head(10))
print("Complete weeks:", len(weeks))
print("Products:", weekly["StockCode"].nunique())

,StockCode,week_start,units
0,10002,2010-12-06,138
1,10002,2010-12-13,41
2,10002,2010-12-20,2
3,10002,2010-12-27,0
4,10002,2011-01-03,73
5,10002,2011-01-10,42
6,10002,2011-01-17,76
7,10002,2011-01-24,17
8,10002,2011-01-31,133
9,10002,2011-02-07,1


Complete weeks: 52
Products: 3890


In [8]:
train_weeks = weeks[:32]
train = weekly.loc[weekly["week_start"].isin(train_weeks)]

summary = train.groupby("StockCode").agg(
    total_units=("units", "sum"),
    active_weeks=("units", lambda s: (s > 0).sum())
)

early_active = set(
    train.loc[
        train["week_start"].isin(train_weeks[:4]) & (train["units"] > 0),
        "StockCode"
    ]
)

late_active = set(
    train.loc[
        train["week_start"].isin(train_weeks[-4:]) & (train["units"] > 0),
        "StockCode"
    ]
)

eligible = summary.loc[
    (summary["active_weeks"] / len(train_weeks) >= 0.70)
    & summary.index.isin(early_active & late_active)
]

selected_products = (
    eligible.sort_values("total_units", ascending=False)
    .head(20)
)

weekly_20 = weekly.loc[
    weekly["StockCode"].isin(selected_products.index)
].copy()

display(selected_products)
print("Selected products:", len(selected_products))

weekly_20.to_csv(output_dir / "weekly_sales_20.csv", index=False)

,total_units,active_weeks
StockCode,,
84077,30207,30
85099B,24180,31
85123A,23208,31
21212,22423,31
22197,21170,31
15036,18761,31
84879,15599,31
22616,15096,31
21977,14747,31


Selected products: 20


In [9]:
forecasts = weekly_20.sort_values(
    ["StockCode", "week_start"]
).copy()

forecasts["last_week"] = (
    forecasts.groupby("StockCode")["units"].shift(1)
)

forecasts["moving_average_4"] = (
    forecasts.groupby("StockCode")["units"]
    .transform(lambda s: s.shift(1).rolling(4).mean())
)

# Assign the same chronological split to every product.
validation_weeks = weeks[32:40]
test_weeks = weeks[40:]

forecasts["split"] = "train"
forecasts.loc[
    forecasts["week_start"].isin(validation_weeks), "split"
] = "validation"
forecasts.loc[
    forecasts["week_start"].isin(test_weeks), "split"
] = "test"

display(forecasts.head(8))

,StockCode,week_start,units,last_week,moving_average_4,split
624,15036,2010-12-06,21,NaN,NaN,train
625,15036,2010-12-13,35,21.0,NaN,train
626,15036,2010-12-20,30,35.0,NaN,train
627,15036,2010-12-27,0,30.0,NaN,train
628,15036,2011-01-03,18,0.0,21.50,train
629,15036,2011-01-10,159,18.0,20.75,train
630,15036,2011-01-17,39,159.0,51.75,train
631,15036,2011-01-24,112,39.0,54.00,train


In [10]:
import numpy as np

validation = forecasts.loc[
    forecasts["split"] == "validation"
].dropna(subset=["last_week", "moving_average_4"]).copy()

results = []

for model in ["last_week", "moving_average_4"]:
    absolute_error = (
        validation["units"] - validation[model]
    ).abs()

    actual_total = validation["units"].sum()

    results.append({
        "model": model,
        "MAE_units": absolute_error.mean(),
        "WAPE_percent": (
            100 * absolute_error.sum() / actual_total
            if actual_total > 0 else np.nan
        ),
        "product_weeks": len(validation)
    })

baseline_results = pd.DataFrame(results)

baseline_mae = baseline_results.loc[
    baseline_results["model"] == "last_week", "MAE_units"
].iloc[0]

baseline_results["improvement_vs_last_week_percent"] = (
    100 * (baseline_mae - baseline_results["MAE_units"]) / baseline_mae
    if baseline_mae > 0 else np.nan
)

display(baseline_results.round(2))

baseline_results.to_csv(
    output_dir / "baseline_validation_results.csv",
    index=False
)

,model,MAE_units,WAPE_percent,product_weeks,improvement_vs_last_week_percent
0,last_week,358.30,69.37,160,0.00
1,moving_average_4,303.01,58.67,160,15.43


In [12]:
features = forecasts.copy()

for lag in [1, 2, 3, 4, 8]:
    features[f"lag_{lag}"] = (
        features.groupby("StockCode")["units"].shift(lag)
    )

for window in [4, 8]:
    features[f"mean_{window}"] = (
        features.groupby("StockCode")["units"]
        .transform(lambda s: s.shift(1).rolling(window).mean())
    )

features["std_4"] = (
    features.groupby("StockCode")["units"]
    .transform(lambda s: s.shift(1).rolling(4).std())
)

features["month"] = features["week_start"].dt.month

numeric_features = [
    "lag_1", "lag_2", "lag_3", "lag_4", "lag_8",
    "mean_4", "mean_8", "std_4", "month"
]

feature_columns = ["StockCode"] + numeric_features
model_data = features.dropna(subset=numeric_features).copy()

In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

def make_model():
    preprocessing = ColumnTransformer([
        ("product", OneHotEncoder(handle_unknown="ignore"), ["StockCode"]),
        ("numeric", "passthrough", numeric_features)
    ])

    return Pipeline([
        ("preprocessing", preprocessing),
        ("model", RandomForestRegressor(
            n_estimators=200,
            max_depth=8,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1
        ))
    ])

validation_predictions = []

for week in validation_weeks:
    training_rows = model_data.loc[
        model_data["week_start"] < week
    ]

    prediction_rows = model_data.loc[
        model_data["week_start"] == week
    ].copy()

    model = make_model()
    model.fit(
        training_rows[feature_columns],
        training_rows["units"]
    )

    prediction_rows["random_forest"] = model.predict(
        prediction_rows[feature_columns]
    ).clip(min=0)

    validation_predictions.append(prediction_rows)

ml_validation = pd.concat(validation_predictions, ignore_index=True)

In [16]:
assert len(ml_validation) == 160
assert not ml_validation[
    ["last_week", "moving_average_4", "random_forest"]
].isna().any().any()

comparison = []

for method in ["last_week", "moving_average_4", "random_forest"]:
    errors = (ml_validation["units"] - ml_validation[method]).abs()
    total_units = ml_validation["units"].sum()

    comparison.append({
        "model": method,
        "MAE_units": errors.mean(),
        "WAPE_percent": (
            100 * errors.sum() / total_units
            if total_units > 0 else np.nan
        ),
        "product_weeks": len(ml_validation)
    })

comparison = pd.DataFrame(comparison)

baseline_mae = comparison.loc[
    comparison["model"] == "last_week", "MAE_units"
].iloc[0]

comparison["improvement_vs_last_week_percent"] = (
    100 * (baseline_mae - comparison["MAE_units"]) / baseline_mae
    if baseline_mae > 0 else np.nan
)

display(comparison.round(2))

comparison.to_csv(
    output_dir / "model_validation_results.csv", index=False
)
ml_validation.to_csv(
    output_dir / "validation_predictions.csv", index=False
)

,model,MAE_units,WAPE_percent,product_weeks,improvement_vs_last_week_percent
0,last_week,358.30,69.37,160,0.00
1,moving_average_4,303.01,58.67,160,15.43
2,random_forest,295.28,57.17,160,17.59


In [17]:
test_predictions = []

for week in test_weeks:
    training_rows = model_data.loc[
        model_data["week_start"] < week
    ]

    prediction_rows = model_data.loc[
        model_data["week_start"] == week
    ].copy()

    model = make_model()
    model.fit(
        training_rows[feature_columns],
        training_rows["units"]
    )

    prediction_rows["random_forest"] = model.predict(
        prediction_rows[feature_columns]
    ).clip(min=0)

    test_predictions.append(prediction_rows)

test_results = pd.concat(test_predictions, ignore_index=True)

methods = ["last_week", "moving_average_4", "random_forest"]

assert len(test_results) == 20 * len(test_weeks)
assert not test_results[methods].isna().any().any()

scores = []

for method in methods:
    errors = (test_results["units"] - test_results[method]).abs()
    total_units = test_results["units"].sum()

    scores.append({
        "model": method,
        "MAE_units": errors.mean(),
        "WAPE_percent": (
            100 * errors.sum() / total_units
            if total_units > 0 else np.nan
        ),
        "product_weeks": len(test_results)
    })

test_scores = pd.DataFrame(scores)

baseline_mae = test_scores.loc[
    test_scores["model"] == "last_week", "MAE_units"
].iloc[0]

test_scores["improvement_vs_last_week_percent"] = (
    100 * (baseline_mae - test_scores["MAE_units"]) / baseline_mae
    if baseline_mae > 0 else np.nan
)

display(test_scores.round(2))

test_results.to_csv(
    output_dir / "test_predictions.csv", index=False
)
test_scores.to_csv(
    output_dir / "test_metrics.csv", index=False
)

,model,MAE_units,WAPE_percent,product_weeks,improvement_vs_last_week_percent
0,last_week,390.79,64.46,240,0.00
1,moving_average_4,315.14,51.99,240,19.36
2,random_forest,323.69,53.40,240,17.17


In [18]:
product_scores = []

for product, rows in test_results.groupby("StockCode"):
    for method in methods:
        errors = (rows["units"] - rows[method]).abs()

        product_scores.append({
            "StockCode": product,
            "model": method,
            "MAE_units": errors.mean()
        })

product_scores = pd.DataFrame(product_scores)

product_comparison = product_scores.pivot(
    index="StockCode",
    columns="model",
    values="MAE_units"
)

product_comparison["lowest_test_MAE_model"] = (
    product_comparison[methods].idxmin(axis=1)
)

display(product_comparison.round(2))

print("\nNumber of products won by each method:")
print(product_comparison["lowest_test_MAE_model"].value_counts())

product_comparison.to_csv(
    output_dir / "test_metrics_by_product.csv"
)

model,last_week,moving_average_4,random_forest,lowest_test_MAE_model
StockCode,,,,
15036,152.17,127.85,257.73,moving_average_4
17003,472.17,407.62,353.08,random_forest
20725,135.58,116.10,124.74,moving_average_4
21212,252.33,226.85,255.66,moving_average_4
21977,219.92,226.60,234.24,last_week
22178,483.92,367.17,294.80,random_forest
22197,797.25,606.90,818.19,moving_average_4
22386,371.08,283.94,300.74,moving_average_4
22469,359.17,318.71,275.41,random_forest



Number of products won by each method:
lowest_test_MAE_model
moving_average_4    10
random_forest        9
last_week            1
Name: count, dtype: int64


In [19]:
import math

# Use training history only to set initial stock and safety stock.
training_means = (
    weekly_20.loc[weekly_20["week_start"].isin(train_weeks)]
    .groupby("StockCode")["units"]
    .mean()
)

def simulate_inventory(
    predictions,
    method,
    training_means,
    lead_time=1,
    initial_stock_weeks=2,
    safety_stock_weeks=0.5
):
    if lead_time < 1:
        raise ValueError("Lead time must be at least one whole week.")

    records = []

    for product, rows in predictions.groupby("StockCode"):
        rows = rows.sort_values("week_start")

        average_demand = float(training_means.loc[product])
        stock = math.ceil(initial_stock_weeks * average_demand)
        safety_stock = math.ceil(safety_stock_weeks * average_demand)
        outstanding_orders = {}

        for step, row in enumerate(rows.itertuples(index=False)):
            opening_stock = stock

            # Receive orders placed in earlier weeks.
            received = outstanding_orders.pop(step, 0)
            stock += received

            forecast = max(0.0, float(getattr(row, method)))
            on_order = sum(outstanding_orders.values())
            inventory_position = stock + on_order

            # Approximate protection-period demand as constant.
            target_stock = math.ceil(
                forecast * (lead_time + 1) + safety_stock
            )

            order_quantity = max(0, target_stock - inventory_position)
            arrival_step = step + lead_time
            outstanding_orders[arrival_step] = (
                outstanding_orders.get(arrival_step, 0)
                + order_quantity
            )

            demand = int(row.units)
            fulfilled = min(stock, demand)
            unmet = demand - fulfilled
            stock -= fulfilled

            # Check conservation of physical stock.
            assert stock == opening_stock + received - fulfilled

            records.append({
                "StockCode": product,
                "week_start": row.week_start,
                "model": method,
                "forecast": forecast,
                "demand": demand,
                "opening_stock": opening_stock,
                "received": received,
                "order_quantity": order_quantity,
                "fulfilled": fulfilled,
                "unmet_units": unmet,
                "closing_stock": stock,
                "outstanding_units": sum(outstanding_orders.values())
            })

    return pd.DataFrame(records)

inventory_results = pd.concat(
    [
        simulate_inventory(test_results, method, training_means)
        for method in methods
    ],
    ignore_index=True
)

In [20]:
inventory_summary = []

for method, rows in inventory_results.groupby("model"):
    total_demand = rows["demand"].sum()

    inventory_summary.append({
        "model": method,
        "unmet_units": rows["unmet_units"].sum(),
        "stockout_product_weeks": (rows["unmet_units"] > 0).sum(),
        "fill_rate_percent": (
            100 * rows["fulfilled"].sum() / total_demand
            if total_demand > 0 else np.nan
        ),
        "average_closing_stock_per_product": rows["closing_stock"].mean()
    })

inventory_summary = pd.DataFrame(inventory_summary)
display(inventory_summary.round(2))

inventory_results.to_csv(
    output_dir / "inventory_simulation.csv", index=False
)
inventory_summary.to_csv(
    output_dir / "inventory_summary.csv", index=False
)

,model,unmet_units,stockout_product_weeks,fill_rate_percent,average_closing_stock_per_product
0,last_week,22783,43,84.34,877.14
1,moving_average_4,27838,52,80.87,432.40
2,random_forest,28933,48,80.11,450.99


In [21]:
scenario_scores = []

for buffer in [0.0, 0.5, 1.0, 2.0]:
    for method in methods:
        simulation = simulate_inventory(
            test_results,
            method,
            training_means,
            lead_time=1,
            initial_stock_weeks=2,
            safety_stock_weeks=buffer
        )

        total_demand = simulation["demand"].sum()

        scenario_scores.append({
            "model": method,
            "safety_stock_weeks": buffer,
            "unmet_units": simulation["unmet_units"].sum(),
            "stockout_product_weeks": (
                simulation["unmet_units"] > 0
            ).sum(),
            "fill_rate_percent": (
                100 * simulation["fulfilled"].sum() / total_demand
                if total_demand > 0 else np.nan
            ),
            "average_closing_stock_per_product": (
                simulation["closing_stock"].mean()
            )
        })

scenario_results = pd.DataFrame(scenario_scores)

display(scenario_results.round(2))

scenario_results.to_csv(
    output_dir / "inventory_scenarios.csv",
    index=False
)

,model,safety_stock_weeks,unmet_units,stockout_product_weeks,fill_rate_percent,average_closing_stock_per_product
0,last_week,0.0,29678,58,79.60,745.30
1,moving_average_4,0.0,38197,76,73.75,307.90
2,random_forest,0.0,38068,66,73.83,312.22
3,last_week,0.5,22783,43,84.34,877.14
4,moving_average_4,0.5,27838,52,80.87,432.40
5,random_forest,0.5,28933,48,80.11,450.99
6,last_week,1.0,16522,33,88.64,1017.22
7,moving_average_4,1.0,19653,38,86.49,578.72
8,random_forest,1.0,21287,41,85.37,610.19
9,last_week,2.0,6837,13,95.30,1351.89


In [22]:
forecasts.to_csv(
    output_dir / "weekly_history.csv",
    index=False
)

In [23]:
forecasts.to_csv(
    output_dir / "weekly_history.csv",
    index=False
)

print(forecasts["week_start"].min())
print(forecasts.groupby("StockCode").size().unique())

2010-12-06 00:00:00
[52]


In [24]:
product_labels = (
    clean.loc[
        clean["StockCode"].isin(selected_products.index)
        & clean["Description"].notna(),
        ["StockCode", "Description"]
    ]
    .assign(Description=lambda x: x["Description"].str.strip())
    .loc[lambda x: x["Description"].ne("")]
    .groupby("StockCode")["Description"]
    .agg(lambda descriptions: descriptions.value_counts().index[0])
    .reset_index()
)

product_labels.to_csv(
    output_dir / "product_labels.csv",
    index=False
)

display(product_labels)

,StockCode,Description
0,15036,ASSORTED COLOURS SILK FAN
1,17003,BROCADE RING PURSE
2,20725,LUNCH BAG RED RETROSPOT
3,21212,PACK OF 72 RETROSPOT CAKE CASES
4,21977,PACK OF 60 PINK PAISLEY CAKE CASES
5,22178,VICTORIAN GLASS HANGING T-LIGHT
6,22197,POPCORN HOLDER
7,22386,JUMBO BAG PINK POLKADOT
8,22469,HEART OF WICKER SMALL
9,22492,MINI PAINT SET VINTAGE


In [25]:
next_week = weekly_20["week_start"].max() + pd.Timedelta(weeks=1)

future_rows = []

for code, rows in weekly_20.groupby("StockCode"):
    sales = rows.sort_values("week_start")["units"]

    future_rows.append({
        "StockCode": code,
        "week_start": next_week,
        "lag_1": sales.iloc[-1],
        "lag_2": sales.iloc[-2],
        "lag_3": sales.iloc[-3],
        "lag_4": sales.iloc[-4],
        "lag_8": sales.iloc[-8],
        "mean_4": sales.tail(4).mean(),
        "mean_8": sales.tail(8).mean(),
        "std_4": sales.tail(4).std(),
        "month": next_week.month
    })

future = pd.DataFrame(future_rows)

# All completed weeks are now available for this next-week forecast.
final_model = make_model()
final_model.fit(
    model_data[feature_columns],
    model_data["units"]
)

future["random_forest"] = final_model.predict(
    future[feature_columns]
).clip(min=0)

future[
    ["StockCode", "week_start", "random_forest"]
].to_csv(
    output_dir / "next_week_forecasts.csv",
    index=False
)

display(future[["StockCode", "week_start", "random_forest"]])

,StockCode,week_start,random_forest
0,15036,2011-12-05,307.219371
1,17003,2011-12-05,378.848045
2,20725,2011-12-05,466.331476
3,21212,2011-12-05,477.145387
4,21977,2011-12-05,395.643393
5,22178,2011-12-05,466.625362
6,22197,2011-12-05,2217.398511
7,22386,2011-12-05,442.746250
8,22469,2011-12-05,409.368438
9,22492,2011-12-05,644.747492
